In [2]:
import polars as pl
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from pathlib import Path
from loguru import logger

BASE_DIR = Path("..")
GOLD_DIR = BASE_DIR / "data" / "3_gold"

# Nạp file Gold Cấp Tuần
df_gold = pl.read_parquet(GOLD_DIR / "weekly_demand_features.parquet")
logger.info(f"Đã nạp thành công {len(df_gold):,} dòng dữ liệu Gold cấp Tuần.")

# Danh sách Features chuẩn nghiệp vụ nâng cao
features = [
    "Lag_1W", "Lag_2W", "Lag_4W", "Rolling_Mean_4W", "Rolling_Std_4W",
    "Demand_Trend_Ratio", "Demand_CV", "Price_Ratio_Vs_Avg", "Avg_UnitPrice",
    "Month", "Quarter", "Year", "Is_Peak_Season"
]
target = "Target_Demand_Weekly"


2026-11-05 22:02:50.205 | INFO     | __main__:<module>:15 - Đã nạp thành công 486,684 dòng dữ liệu Gold cấp Tuần.


In [3]:
# Lọc Top 20 SKU chủ lực (quy tắc 80/20 Pareto)
top_skus = (
    df_gold.group_by("StockCode")
    .agg(pl.col("Target_Demand_Weekly").sum().alias("Total_Demand"))
    .sort("Total_Demand", descending=True)
    .head(20)
    .select("StockCode")
)

df_sub = df_gold.join(top_skus, on="StockCode", how="inner")

# Chia Train (80% quá khứ) và Test (20% gần nhất) theo mốc Tuần
weeks = df_sub.select("Week_Start").unique().sort("Week_Start")
split_idx = int(len(weeks) * 0.8)
split_week = weeks[split_idx, "Week_Start"]

train_df = df_sub.filter(pl.col("Week_Start") < split_week)
test_df = df_sub.filter(pl.col("Week_Start") >= split_week)

X_train = train_df.select(features).to_pandas()
y_train = train_df.select(target).to_pandas().values.ravel()

X_test = test_df.select(features).to_pandas()
y_test = test_df.select(target).to_pandas().values.ravel()

logger.success(f"Tập Train: {X_train.shape[0]:,} mẫu | Tập Test: {X_test.shape[0]:,} mẫu")


2026-11-05 22:02:50.393 | SUCCESS  | __main__:<module>:26 - Tập Train: 1,580 mẫu | Tập Test: 400 mẫu


In [4]:
# 1. Baseline Model (Naive: Lấy đúng nhu cầu tuần trước Lag_1W)
y_pred_base = test_df.select("Lag_1W").to_pandas().values.ravel()

# 2. Random Forest Model
rf_model = RandomForestRegressor(n_estimators=150, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

# 3. XGBoost Model với Log Transformation & Hyperparameters tối ưu
y_train_log = np.log1p(y_train)
xgb_model = xgb.XGBRegressor(
    n_estimators=200,
    learning_rate=0.03,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train_log)

# Dự báo và nới ngược Log (expm1)
y_pred_xgb_log = xgb_model.predict(X_test)
y_pred_xgb = np.clip(np.expm1(y_pred_xgb_log), a_min=0, a_max=None)

logger.success("Huấn luyện hoàn tất 3 mô hình!")


2026-11-05 22:02:51.359 | SUCCESS  | __main__:<module>:26 - Huấn luyện hoàn tất 3 mô hình!


In [5]:

# ============================================================
# CELL 4: EVALUATION - MULTI-SKU DEMAND FORECASTING
# ============================================================

import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

# 1. Prepare evaluation data
test_eval_df = test_df.select(["StockCode", target]).to_pandas()

test_eval_df["Y_Actual"] = test_eval_df[target].to_numpy()
test_eval_df["Y_Pred_Base"] = np.asarray(y_pred_base).reshape(-1)
test_eval_df["Y_Pred_RF"] = np.asarray(y_pred_rf).reshape(-1)
test_eval_df["Y_Pred_XGB"] = np.asarray(y_pred_xgb).reshape(-1)

assert len(test_eval_df) == len(y_pred_xgb), "Prediction length mismatch"
assert (test_eval_df["Y_Actual"] >= 0).all(), "Negative actual demand"

# 2. Evaluation function
def evaluate_model(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    mae = mean_absolute_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)

    denominator = np.sum(y_true)
    wape = np.sum(np.abs(y_true - y_pred)) / denominator * 100 if denominator > 0 else np.nan

    bias = np.sum(y_pred - y_true) / denominator * 100 if denominator > 0 else np.nan

    return {
        "MAE": mae,
        "RMSE": rmse,
        "WAPE (%)": wape,
        "Forecast Score (%)": 100 - wape if pd.notna(wape) else np.nan,
        "Bias (%)": bias
    }

# 3. Global model comparison - KEEP ALL TEST DATA
predictions = {
    "Baseline": "Y_Pred_Base",
    "Random Forest": "Y_Pred_RF",
    "XGBoost": "Y_Pred_XGB"
}

global_results = []

for model_name, pred_col in predictions.items():
    metrics = evaluate_model(
        test_eval_df["Y_Actual"],
        test_eval_df[pred_col]
    )
    global_results.append({"Model": model_name, **metrics})

global_results_df = pd.DataFrame(global_results)

print("GLOBAL TEST PERFORMANCE")
display(global_results_df.round(3))

# 4. Per-SKU evaluation
sku_results = []

for sku, group in test_eval_df.groupby("StockCode"):
    metrics = evaluate_model(
        group["Y_Actual"],
        group["Y_Pred_XGB"]
    )
    sku_results.append({
        "StockCode": sku,
        "Actual_Demand": group["Y_Actual"].sum(),
        "XGB_MAE": metrics["MAE"],
        "XGB_RMSE": metrics["RMSE"],
        "XGB_WAPE": metrics["WAPE (%)"],
        "XGB_Bias": metrics["Bias (%)"]
    })

sku_results_df = pd.DataFrame(sku_results)

print("PER-SKU PERFORMANCE")
display(sku_results_df.sort_values("XGB_WAPE", ascending=False).round(3))

# 5. Macro evaluation
valid_skus = sku_results_df.dropna(subset=["XGB_WAPE"])

print("\nMACRO / MEDIAN EVALUATION")
print(f"Macro WAPE: {valid_skus['XGB_WAPE'].mean():.2f}%")
print(f"Median SKU WAPE: {valid_skus['XGB_WAPE'].median():.2f}%")
print(f"Median SKU Forecast Score: {100 - valid_skus['XGB_WAPE'].median():.2f}%")

# 6. Demand and prediction diagnostics
test_eval_df["Residual"] = (
    test_eval_df["Y_Actual"] - test_eval_df["Y_Pred_XGB"]
)

print("\nERROR DIAGNOSTICS")
print(f"Under-forecast rows: {(test_eval_df['Residual'] > 0).sum()}")
print(f"Over-forecast rows: {(test_eval_df['Residual'] < 0).sum()}")
print(f"Mean residual: {test_eval_df['Residual'].mean():.3f}")
print(f"Maximum absolute error: {test_eval_df['Residual'].abs().max():.3f}")


GLOBAL TEST PERFORMANCE


,Model,MAE,RMSE,WAPE (%),Forecast Score (%),Bias (%)
0,Baseline,595.832,4102.427,77.235,22.765,-27.825
1,Random Forest,554.440,4087.079,71.869,28.131,-19.511
2,XGBoost,511.182,4091.297,66.262,33.738,-43.750


PER-SKU PERFORMANCE


,StockCode,Actual_Demand,XGB_MAE,XGB_RMSE,XGB_WAPE,XGB_Bias
13,23843,80995,4049.847,18111.010,100.002,-99.997
0,15036,4480,205.342,265.750,91.670,10.188
1,17003,7566,303.644,432.595,80.265,-13.289
12,23166,2251,87.861,106.971,78.064,35.000
4,21213,6415,207.278,265.893,64.623,9.618
5,21915,13245,424.788,721.961,64.143,-33.334
11,22616,9374,299.448,445.100,63.889,-27.289
8,22197,34637,1069.114,1497.259,61.732,-58.283
10,22492,11909,354.968,467.153,59.613,-21.013
17,84991,7334,211.649,264.861,57.717,-5.749



MACRO / MEDIAN EVALUATION
Macro WAPE: 58.98%
Median SKU WAPE: 57.31%
Median SKU Forecast Score: 42.69%

ERROR DIAGNOSTICS
Under-forecast rows: 181
Over-forecast rows: 219
Mean residual: 337.508
Maximum absolute error: 80994.893


In [6]:

# ============================================================
# DIAGNOSTIC 1: FIND LARGEST FORECAST ERRORS
# ============================================================

error_df = test_eval_df.copy()

error_df["Absolute_Error"] = (
    error_df["Y_Actual"] - error_df["Y_Pred_XGB"]
).abs()

error_df["Residual"] = (
    error_df["Y_Actual"] - error_df["Y_Pred_XGB"]
)

error_df["Error_Percent"] = np.where(
    error_df["Y_Actual"] > 0,
    error_df["Absolute_Error"] / error_df["Y_Actual"] * 100,
    np.nan
)

top_errors = error_df.sort_values(
    "Absolute_Error", ascending=False
).head(20)

display(top_errors[
    ["StockCode", "Y_Actual", "Y_Pred_XGB",
     "Residual", "Absolute_Error", "Error_Percent"]
].round(2))


,StockCode,Y_Actual,Y_Pred_XGB,Residual,Absolute_Error,Error_Percent
279,23843,80995,0.110000,80994.89,80994.89,100.00
293,84077,5328,647.280029,4680.72,4680.72,87.85
179,22197,4456,1030.630005,3425.37,3425.37,76.87
321,84879,3800,379.239990,3420.76,3420.76,90.02
175,22197,3900,740.229980,3159.77,3159.77,81.02
178,22197,3275,1025.020020,2249.98,2249.98,68.70
114,21915,2652,446.489990,2205.51,2205.51,83.16
370,85099B,2709,759.679993,1949.32,1949.32,71.96
173,22197,2775,836.590027,1938.41,1938.41,69.85
176,22197,3145,1212.630005,1932.37,1932.37,61.44


In [7]:

# ============================================================
# DIAGNOSTIC 2: SKU PERFORMANCE
# ============================================================

sku_analysis = (
    error_df.groupby("StockCode")
    .agg(
        Actual_Total=("Y_Actual", "sum"),
        Predicted_Total=("Y_Pred_XGB", "sum"),
        MAE=("Absolute_Error", "mean"),
        Total_Error=("Absolute_Error", "sum"),
        Mean_Residual=("Residual", "mean"),
        Test_Weeks=("Y_Actual", "count")
    )
    .reset_index()
)

sku_analysis["WAPE"] = np.where(
    sku_analysis["Actual_Total"] > 0,
    sku_analysis["Total_Error"] / sku_analysis["Actual_Total"] * 100,
    np.nan
)

sku_analysis["Bias"] = np.where(
    sku_analysis["Actual_Total"] > 0,
    (sku_analysis["Predicted_Total"] -
     sku_analysis["Actual_Total"]) /
    sku_analysis["Actual_Total"] * 100,
    np.nan
)

sku_analysis["Demand_Share"] = (
    sku_analysis["Actual_Total"] /
    sku_analysis["Actual_Total"].sum() * 100
)

print("TOP 10 SKU WITH HIGHEST WAPE")
display(
    sku_analysis.sort_values("WAPE", ascending=False)
    .head(10).round(2)
)

print("TOP 10 SKU WITH HIGHEST CONTRIBUTION TO TOTAL ERROR")
display(
    sku_analysis.sort_values("Total_Error", ascending=False)
    .head(10).round(2)
)


TOP 10 SKU WITH HIGHEST WAPE


,StockCode,Actual_Total,Predicted_Total,MAE,Total_Error,Mean_Residual,Test_Weeks,WAPE,Bias,Demand_Share
13,23843,80995,2.160000,4049.85,80996.95,4049.64,20,100.00,-100.00,26.25
0,15036,4480,4936.430176,205.34,4106.83,-22.82,20,91.67,10.19,1.45
1,17003,7566,6560.580078,303.64,6072.87,50.27,20,80.27,-13.29,2.45
12,23166,2251,3038.850098,87.86,1757.22,-39.39,20,78.06,35.00,0.73
4,21213,6415,7031.990234,207.28,4145.57,-30.85,20,64.62,9.62,2.08
5,21915,13245,8829.919922,424.79,8495.77,220.75,20,64.14,-33.33,4.29
11,22616,9374,6815.919922,299.45,5988.96,127.90,20,63.89,-27.29,3.04
8,22197,34637,14449.469727,1069.11,21382.28,1009.38,20,61.73,-58.28,11.22
10,22492,11909,9406.509766,354.97,7099.36,125.12,20,59.61,-21.01,3.86
17,84991,7334,6912.390137,211.65,4232.98,21.08,20,57.72,-5.75,2.38


TOP 10 SKU WITH HIGHEST CONTRIBUTION TO TOTAL ERROR


,StockCode,Actual_Total,Predicted_Total,MAE,Total_Error,Mean_Residual,Test_Weeks,WAPE,Bias,Demand_Share
13,23843,80995,2.160000,4049.85,80996.95,4049.64,20,100.00,-100.00,26.25
8,22197,34637,14449.469727,1069.11,21382.28,1009.38,20,61.73,-58.28,11.22
18,85099B,22869,17688.859375,521.06,10421.30,259.01,20,45.57,-22.65,7.41
14,84077,20705,12557.780273,509.39,10187.74,407.36,20,49.20,-39.35,6.71
16,84879,19439,13539.379883,489.55,9791.00,294.98,20,50.37,-30.35,6.30
5,21915,13245,8829.919922,424.79,8495.77,220.75,20,64.14,-33.33,4.29
10,22492,11909,9406.509766,354.97,7099.36,125.12,20,59.61,-21.01,3.86
1,17003,7566,6560.580078,303.64,6072.87,50.27,20,80.27,-13.29,2.45
11,22616,9374,6815.919922,299.45,5988.96,127.90,20,63.89,-27.29,3.04
19,85123A,12747,11238.719727,286.71,5734.24,75.41,20,44.99,-11.83,4.13


In [8]:

# ============================================================
# DIAGNOSTIC 3: DEMAND DISTRIBUTION
# ============================================================

print("Actual demand statistics:")
print(error_df["Y_Actual"].describe())

zero_ratio = (
    error_df["Y_Actual"].eq(0).mean() * 100
)

print(f"\nZero-demand ratio: {zero_ratio:.2f}%")

print("\nNegative predictions:")
print((error_df["Y_Pred_XGB"] < 0).sum())

print("\nNegative actual values:")
print((error_df["Y_Actual"] < 0).sum())


Actual demand statistics:
count      400.000000
mean       771.455000
std       4071.723625
min          0.000000
25%        201.500000
50%        398.000000
75%        742.250000
max      80995.000000
Name: Y_Actual, dtype: float64

Zero-demand ratio: 4.75%

Negative predictions:
0

Negative actual values:
0


In [9]:

# ============================================================
# DIAGNOSTIC 4: SKU HISTORY CHECK
# ============================================================

target_skus = ["23843", "22197", "84077", "21915"]

# Convert StockCode to string for safe comparison
train_check = train_df.to_pandas() if hasattr(train_df, "to_pandas") else train_df.copy()
test_check = test_df.to_pandas() if hasattr(test_df, "to_pandas") else test_df.copy()

train_check["StockCode"] = train_check["StockCode"].astype(str)
test_check["StockCode"] = test_check["StockCode"].astype(str)

for sku in target_skus:
    tr = train_check[train_check["StockCode"] == sku]
    te = test_check[test_check["StockCode"] == sku]

    print(f"\n========== SKU {sku} ==========")

    print("Train rows:", len(tr))
    print("Test rows:", len(te))

    if len(tr) > 0:
        print("Train demand total:", tr[target].sum())
        print("Train demand max:", tr[target].max())
        print("Train demand mean:", tr[target].mean())

    if len(te) > 0:
        print("Test demand total:", te[target].sum())
        print("Test demand max:", te[target].max())
        print("Test demand mean:", te[target].mean())
        print("Test demand median:", te[target].median())



========== SKU 23843 ==========
Train rows: 79
Test rows: 20
Train demand total: 0
Train demand max: 0
Train demand mean: 0.0
Test demand total: 80995
Test demand max: 80995
Test demand mean: 4049.75
Test demand median: 0.0

========== SKU 22197 ==========
Train rows: 79
Test rows: 20
Train demand total: 53764
Train demand max: 4562
Train demand mean: 680.5569620253165
Test demand total: 34637
Test demand max: 4456
Test demand mean: 1731.85
Test demand median: 1160.0

========== SKU 84077 ==========
Train rows: 79
Test rows: 20
Train demand total: 86394
Train demand max: 7063
Train demand mean: 1093.5949367088608
Test demand total: 20705
Test demand max: 5328
Test demand mean: 1035.25
Test demand median: 944.5

========== SKU 21915 ==========
Train rows: 79
Test rows: 20
Train demand total: 24945
Train demand max: 3432
Train demand mean: 315.75949367088606
Test demand total: 13245
Test demand max: 2652
Test demand mean: 662.25
Test demand median: 358.5


In [10]:

# ============================================================
# DIAGNOSTIC 5: SKU 23843 WEEKLY DEMAND
# ============================================================

sku = "23843"

sku_history = test_check[
    test_check["StockCode"] == sku
].copy()

print("SKU 23843 - TEST HISTORY")

display(sku_history)


SKU 23843 - TEST HISTORY


,Week_Start,StockCode,Weekly_Demand,Weekly_Revenue,Avg_UnitPrice,Description,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W,Month,Quarter,Year,Is_Peak_Season,Demand_Trend_Ratio,Demand_CV,Price_Ratio_Vs_Avg
260,2011-07-18,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,7,3,2011,0,0.0,0.0,0.0
261,2011-07-25,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,7,3,2011,0,0.0,0.0,0.0
262,2011-08-01,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,8,3,2011,0,0.0,0.0,0.0
263,2011-08-08,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,8,3,2011,0,0.0,0.0,0.0
264,2011-08-15,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,8,3,2011,0,0.0,0.0,0.0
265,2011-08-22,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,8,3,2011,0,0.0,0.0,0.0
266,2011-08-29,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,8,3,2011,0,0.0,0.0,0.0
267,2011-09-05,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,9,3,2011,0,0.0,0.0,0.0
268,2011-09-12,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,9,3,2011,0,0.0,0.0,0.0
269,2011-09-19,23843,0,0.0,0.0,UNKNOWN,0,0,0,0,0.0,0.0,9,3,2011,0,0.0,0.0,0.0


In [11]:
print("Variables related to model data:")

for name in sorted(globals()):
    if any(key in name.lower() for key in ["train", "test", "val"]):
        obj = globals()[name]
        if hasattr(obj, "shape"):
            print(f"{name}: {obj.shape}")

Variables related to model data:
X_test: (400, 13)
X_train: (1580, 13)
test_check: (400, 19)
test_df: (400, 19)
test_eval_df: (400, 7)
train_check: (1580, 19)
train_df: (1580, 19)
valid_skus: (20, 6)
y_test: (400,)
y_train: (1580,)
y_train_log: (1580,)


In [12]:

# ============================================================
# CHECK TARGET ALIGNMENT
# ============================================================

test_check = test_df.to_pandas() if hasattr(test_df, "to_pandas") else test_df.copy()

test_check["Week_Start"] = pd.to_datetime(test_check["Week_Start"])
test_check = test_check.sort_values(["StockCode", "Week_Start"])

sku = "23843"
cols = [
    "Week_Start", "StockCode",
    "Weekly_Demand", "Target_Demand_Weekly",
    "Lag_1W", "Lag_2W", "Lag_4W"
]

display(
    test_check[test_check["StockCode"].astype(str) == sku][cols]
)


,Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W
260,2011-07-18,23843,0,0,0,0,0
261,2011-07-25,23843,0,0,0,0,0
262,2011-08-01,23843,0,0,0,0,0
263,2011-08-08,23843,0,0,0,0,0
264,2011-08-15,23843,0,0,0,0,0
265,2011-08-22,23843,0,0,0,0,0
266,2011-08-29,23843,0,0,0,0,0
267,2011-09-05,23843,0,0,0,0,0
268,2011-09-12,23843,0,0,0,0,0
269,2011-09-19,23843,0,0,0,0,0


In [13]:

# ============================================================
# VERIFY TARGET = NEXT WEEK DEMAND
# ============================================================

check_df = test_check.copy()

check_df["Next_Week_Demand"] = (
    check_df.groupby("StockCode")["Weekly_Demand"].shift(-1)
)

check_df["Target_Match"] = np.isclose(
    check_df["Target_Demand_Weekly"],
    check_df["Next_Week_Demand"],
    equal_nan=True
)

print("Target alignment:")
print(check_df["Target_Match"].value_counts(dropna=False))

print("\nRows with mismatch:")
display(
    check_df[
        ~check_df["Target_Match"]
    ][[
        "Week_Start", "StockCode",
        "Weekly_Demand", "Target_Demand_Weekly",
        "Next_Week_Demand"
    ]].head(20)
)


Target alignment:
Target_Match
True     380
False     20
Name: count, dtype: int64

Rows with mismatch:


,Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly,Next_Week_Demand
19,2011-11-28,15036,143,14,NaN
39,2011-11-28,17003,130,277,NaN
59,2011-11-28,20725,331,264,NaN
79,2011-11-28,21212,546,266,NaN
99,2011-11-28,21213,210,106,NaN
119,2011-11-28,21915,1120,2,NaN
139,2011-11-28,21977,239,207,NaN
159,2011-11-28,22178,755,656,NaN
179,2011-11-28,22197,3275,4456,NaN
199,2011-11-28,22386,436,102,NaN


In [14]:
import pandas as pd
import numpy as np

full_check = pd.concat([
    train_df.to_pandas() if hasattr(train_df, "to_pandas") else train_df,
    test_df.to_pandas() if hasattr(test_df, "to_pandas") else test_df
], ignore_index=True)

full_check["Week_Start"] = pd.to_datetime(full_check["Week_Start"])
full_check = full_check.sort_values(["StockCode", "Week_Start"])

full_check["Next_Week_Demand"] = (
    full_check.groupby("StockCode")["Weekly_Demand"].shift(-1)
)

full_check["Target_Match"] = np.isclose(
    full_check["Target_Demand_Weekly"],
    full_check["Next_Week_Demand"],
    equal_nan=True
)

print(full_check["Target_Match"].value_counts(dropna=False))

display(
    full_check[
        (full_check["StockCode"] == 23843)
    ][[
        "Week_Start",
        "Weekly_Demand",
        "Target_Demand_Weekly",
        "Next_Week_Demand",
        "Target_Match"
    ]].tail(25)
)

Target_Match
True     1960
False      20
Name: count, dtype: int64


,Week_Start,Weekly_Demand,Target_Demand_Weekly,Next_Week_Demand,Target_Match


In [15]:

for name, obj in globals().copy().items():
    if hasattr(obj, "columns") and hasattr(obj, "shape"):
        print(name, obj.shape, list(obj.columns)[:12])

df_gold (486684, 19) ['Week_Start', 'StockCode', 'Weekly_Demand', 'Weekly_Revenue', 'Avg_UnitPrice', 'Description', 'Target_Demand_Weekly', 'Lag_1W', 'Lag_2W', 'Lag_4W', 'Rolling_Mean_4W', 'Rolling_Std_4W']
top_skus (20, 1) ['StockCode']
df_sub (1980, 19) ['Week_Start', 'StockCode', 'Weekly_Demand', 'Weekly_Revenue', 'Avg_UnitPrice', 'Description', 'Target_Demand_Weekly', 'Lag_1W', 'Lag_2W', 'Lag_4W', 'Rolling_Mean_4W', 'Rolling_Std_4W']
weeks (99, 1) ['Week_Start']
train_df (1580, 19) ['Week_Start', 'StockCode', 'Weekly_Demand', 'Weekly_Revenue', 'Avg_UnitPrice', 'Description', 'Target_Demand_Weekly', 'Lag_1W', 'Lag_2W', 'Lag_4W', 'Rolling_Mean_4W', 'Rolling_Std_4W']
test_df (400, 19) ['Week_Start', 'StockCode', 'Weekly_Demand', 'Weekly_Revenue', 'Avg_UnitPrice', 'Description', 'Target_Demand_Weekly', 'Lag_1W', 'Lag_2W', 'Lag_4W', 'Rolling_Mean_4W', 'Rolling_Std_4W']
X_train (1580, 13) ['Lag_1W', 'Lag_2W', 'Lag_4W', 'Rolling_Mean_4W', 'Rolling_Std_4W', 'Demand_Trend_Ratio', 'Demand_CV

In [21]:
sku = "23843"

sku_raw = (
    df_gold
    .filter(pl.col("StockCode") == sku)
    .sort("Week_Start")
    .with_columns(
        pl.col("Weekly_Demand")
        .shift(-1)
        .alias("Next_Week_Demand")
    )
    .with_columns(
        (pl.col("Target_Demand_Weekly") == pl.col("Next_Week_Demand"))
        .alias("Target_Match")
    )
)

display(
    sku_raw.select([
        "Week_Start",
        "Weekly_Demand",
        "Target_Demand_Weekly",
        "Next_Week_Demand",
        "Target_Match"
    ]).tail(30)
)
print(
    df_gold.select([
        pl.col("Week_Start").min().alias("Min_Week"),
        pl.col("Week_Start").max().alias("Max_Week")
    ])
)

Week_Start,Weekly_Demand,Target_Demand_Weekly,Next_Week_Demand,Target_Match
datetime[μs],i64,i64,i64,bool
2011-05-09 00:00:00,0,0,0,true
2011-05-16 00:00:00,0,0,0,true
2011-05-23 00:00:00,0,0,0,true
2011-05-30 00:00:00,0,0,0,true
2011-06-06 00:00:00,0,0,0,true
…,…,…,…,…
2011-10-31 00:00:00,0,0,0,true
2011-11-07 00:00:00,0,0,0,true
2011-11-14 00:00:00,0,0,0,true


shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ Min_Week            ┆ Max_Week            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-01-04 00:00:00 ┆ 2011-11-28 00:00:00 │
└─────────────────────┴─────────────────────┘


In [22]:
display(
    df_gold
    .filter(
        (pl.col("Target_Demand_Weekly") >= 10000) |
        (pl.col("Weekly_Demand") >= 10000)
    )
    .select([
        "Week_Start",
        "StockCode",
        "Weekly_Demand",
        "Target_Demand_Weekly",
        "Description"
    ])
    .sort("Target_Demand_Weekly", descending=True)
)

Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly,Description
datetime[μs],str,i64,i64,str
2011-11-28 00:00:00,"""23843""",0,80995,"""UNKNOWN"""
2011-01-10 00:00:00,"""23166""",0,74215,"""UNKNOWN"""
2010-02-08 00:00:00,"""37410""",0,19165,"""BLACK AND WHITE PAISLEY FLOWER…"
2010-03-08 00:00:00,"""21091""",3,12976,"""SET/6 WOODLAND PAPER PLATES"""
2010-03-08 00:00:00,"""21099""",24,12960,"""SET/6 STRAWBERRY PAPER CUPS"""
…,…,…,…,…
2010-03-15 00:00:00,"""21091""",12976,0,"""SET/6 WOODLAND PAPER PLATES"""
2010-03-15 00:00:00,"""21092""",12481,0,"""SET/6 STRAWBERRY PAPER PLATES"""
2010-03-15 00:00:00,"""21099""",12960,0,"""SET/6 STRAWBERRY PAPER CUPS"""


In [23]:
display(
    df_gold
    .filter(pl.col("StockCode") == "23843")
    .select([
        pl.col("Weekly_Demand").sum().alias("Total_Weekly_Demand"),
        pl.col("Weekly_Demand").max().alias("Max_Weekly_Demand"),
        pl.col("Target_Demand_Weekly").sum().alias("Total_Target"),
        pl.col("Target_Demand_Weekly").max().alias("Max_Target"),
        pl.len().alias("Number_of_Weeks")
    ])
)

Total_Weekly_Demand,Max_Weekly_Demand,Total_Target,Max_Target,Number_of_Weeks
i64,i64,i64,i64,u32
0,0,80995,80995,99


In [24]:
target_check = (
    df_gold
    .sort(["StockCode", "Week_Start"])
    .with_columns(
        pl.col("Weekly_Demand")
        .shift(-1)
        .over("StockCode")
        .alias("Next_Week_Demand")
    )
    .with_columns(
        (pl.col("Target_Demand_Weekly") == pl.col("Next_Week_Demand"))
        .alias("Target_Match")
    )
)

# Chỉ đánh giá các dòng có tuần kế tiếp
print(
    target_check
    .filter(pl.col("Next_Week_Demand").is_not_null())
    .group_by("Target_Match")
    .len()
)

shape: (1, 2)
┌──────────────┬────────┐
│ Target_Match ┆ len    │
│ ---          ┆ ---    │
│ bool         ┆ u32    │
╞══════════════╪════════╡
│ true         ┆ 481768 │
└──────────────┴────────┘


In [25]:
display(
    target_check
    .filter(
        pl.col("Next_Week_Demand").is_not_null() &
        ~pl.col("Target_Match")
    )
    .select([
        "Week_Start", "StockCode",
        "Weekly_Demand",
        "Target_Demand_Weekly",
        "Next_Week_Demand"
    ])
    .head(30)
)

Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly,Next_Week_Demand
datetime[μs],str,i64,i64,i64


In [26]:
from pathlib import Path

for path in Path("../data").rglob("*"):
    if path.is_file():
        print(path)

../data/2_silver/sheet_2010_2011_cleaned.parquet
../data/2_silver/sheet_2009_2010_cleaned.parquet
../data/1_bronze/sheet_2010_2011_raw.parquet
../data/1_bronze/sheet_2009_2010_raw.parquet
../data/raw/online_retail_II.xlsx
../data/3_gold/daily_demand_features.parquet
../data/3_gold/weekly_demand_features.parquet


In [28]:
silver_files = [
    Path("../data/2_silver/sheet_2009_2010_cleaned.parquet"),
    Path("../data/2_silver/sheet_2010_2011_cleaned.parquet")
]

for path in silver_files:
    df = pl.read_parquet(path)

    print(f"\nFile: {path.name}")
    print("Shape:", df.shape)
    print("Schema:", df.schema)

    print(
        df.select([
            pl.col("InvoiceDate").min().alias("Min_InvoiceDate"),
            pl.col("InvoiceDate").max().alias("Max_InvoiceDate")
        ])
    )


File: sheet_2009_2010_cleaned.parquet
Shape: (511565, 8)
Schema: Schema({'Invoice': String, 'StockCode': String, 'Description': String, 'Quantity': Int64, 'InvoiceDate': Datetime(time_unit='us', time_zone=None), 'UnitPrice': Float64, 'CustomerID': String, 'Country': String})
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ Min_InvoiceDate     ┆ Max_InvoiceDate     │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2009-12-01 07:45:00 ┆ 2010-12-09 20:01:00 │
└─────────────────────┴─────────────────────┘

File: sheet_2010_2011_cleaned.parquet
Shape: (530105, 8)
Schema: Schema({'Invoice': String, 'StockCode': String, 'Description': String, 'Quantity': Int64, 'InvoiceDate': Datetime(time_unit='us', time_zone=None), 'UnitPrice': Float64, 'CustomerID': String, 'Country': String})
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ Min_InvoiceDate     ┆ Max_InvoiceDate     │
│ --

In [29]:
sku = "23843"

for path in silver_files:
    df = pl.read_parquet(path)

    result = (
        df.filter(pl.col("StockCode").cast(pl.Utf8) == sku)
        .sort("InvoiceDate")
    )

    print(f"\n========== {path.name} ==========")
    print("Number of transactions:", result.height)

    display(result)


========== sheet_2009_2010_cleaned.parquet ==========
Number of transactions: 0


Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str



========== sheet_2010_2011_cleaned.parquet ==========
Number of transactions: 1


Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str
"""581483""","""23843""","""PAPER CRAFT , LITTLE BIRDIE""",80995,2011-12-09 09:15:00,2.08,"""16446""","""United Kingdom"""


In [30]:
for path in silver_files:
    df = pl.read_parquet(path)

    result = (
        df.filter(pl.col("StockCode").cast(pl.Utf8) == "23843")
        .group_by("StockCode")
        .agg([
            pl.len().alias("Transaction_Rows"),
            pl.col("Quantity").sum().alias("Total_Quantity"),
            pl.col("Quantity").max().alias("Max_Quantity"),
            pl.col("InvoiceDate").min().alias("First_Transaction"),
            pl.col("InvoiceDate").max().alias("Last_Transaction")
        ])
    )

    print(f"\n{path.name}")
    display(result)


sheet_2009_2010_cleaned.parquet


StockCode,Transaction_Rows,Total_Quantity,Max_Quantity,First_Transaction,Last_Transaction
str,u32,i64,i64,datetime[μs],datetime[μs]



sheet_2010_2011_cleaned.parquet


StockCode,Transaction_Rows,Total_Quantity,Max_Quantity,First_Transaction,Last_Transaction
str,u32,i64,i64,datetime[μs],datetime[μs]
"""23843""",1,80995,80995,2011-12-09 09:15:00,2011-12-09 09:15:00


In [32]:
for file in [
    GOLD_DIR / "daily_demand_features.parquet",
    GOLD_DIR / "weekly_demand_features.parquet"
]:
    df = pl.read_parquet(file)
    date_col = "Date" if "Date" in df.columns else "Week_Start"

    print(f"\n{file.name} | Shape: {df.shape}")
    print(df.select([
        pl.col(date_col).min().alias("Min_Date"),
        pl.col(date_col).max().alias("Max_Date")
    ]))


daily_demand_features.parquet | Shape: (2866028, 17)
shape: (1, 2)
┌────────────┬────────────┐
│ Min_Date   ┆ Max_Date   │
│ ---        ┆ ---        │
│ date       ┆ date       │
╞════════════╪════════════╡
│ 2009-12-16 ┆ 2011-12-01 │
└────────────┴────────────┘

weekly_demand_features.parquet | Shape: (486684, 19)
shape: (1, 2)
┌─────────────────────┬─────────────────────┐
│ Min_Date            ┆ Max_Date            │
│ ---                 ┆ ---                 │
│ datetime[μs]        ┆ datetime[μs]        │
╞═════════════════════╪═════════════════════╡
│ 2010-01-04 00:00:00 ┆ 2011-11-28 00:00:00 │
└─────────────────────┴─────────────────────┘


In [33]:
daily = pl.read_parquet(GOLD_DIR / "daily_demand_features.parquet")

display(
    daily
    .filter(
        (pl.col("StockCode") == "23843") &
        (pl.col("Date") >= pl.datetime(2011, 12, 1))
    )
    .sort("Date")
)

Date,StockCode,Daily_Demand,Daily_Revenue,Avg_UnitPrice,Description,Day,DayOfWeek,Month,Quarter,Year,Is_Weekend,Lag_1,Lag_7,Lag_14,Target_Demand_7D,Rolling_Mean_7D
date,str,i64,f64,f64,str,i8,i8,i8,i8,i32,i32,i64,i64,i64,i64,f64
2011-12-01,"""23843""",0,0.0,0.0,"""UNKNOWN""",1,4,12,4,2011,0,0,0,0,80995,0.0


In [34]:
display(
    df_gold
    .filter(pl.col("Week_Start") >= pl.datetime(2011, 11, 1))
    .select([
        "Week_Start",
        "StockCode",
        "Weekly_Demand",
        "Target_Demand_Weekly"
    ])
    .sort(["StockCode", "Week_Start"])
    .tail(30)
)

Week_Start,StockCode,Weekly_Demand,Target_Demand_Weekly
datetime[μs],str,i64,i64
2011-11-21 00:00:00,"""gift_0001_10""",0,0
2011-11-28 00:00:00,"""gift_0001_10""",0,1
2011-11-07 00:00:00,"""gift_0001_20""",0,0
2011-11-14 00:00:00,"""gift_0001_20""",0,0
2011-11-21 00:00:00,"""gift_0001_20""",0,0
…,…,…,…
2011-11-28 00:00:00,"""gift_0001_80""",0,0
2011-11-07 00:00:00,"""m""",0,0
2011-11-14 00:00:00,"""m""",0,0


In [35]:
daily = pl.read_parquet(GOLD_DIR / "daily_demand_features.parquet")

print("Daily shape:", daily.shape)
print("Date range:")
display(daily.select([
    pl.col("Date").min().alias("Min_Date"),
    pl.col("Date").max().alias("Max_Date")
]))

display(
    daily
    .filter(pl.col("StockCode") == "23843")
    .sort("Date")
    .tail(20)
)

Daily shape: (2866028, 17)
Date range:


Min_Date,Max_Date
date,date
2009-12-16,2011-12-01


Date,StockCode,Daily_Demand,Daily_Revenue,Avg_UnitPrice,Description,Day,DayOfWeek,Month,Quarter,Year,Is_Weekend,Lag_1,Lag_7,Lag_14,Target_Demand_7D,Rolling_Mean_7D
date,str,i64,f64,f64,str,i8,i8,i8,i8,i32,i32,i64,i64,i64,i64,f64
2011-11-09,"""23843""",0,0.0,0.0,"""UNKNOWN""",9,3,11,4,2011,0,0,0,0,0,0.0
2011-11-10,"""23843""",0,0.0,0.0,"""UNKNOWN""",10,4,11,4,2011,0,0,0,0,0,0.0
2011-11-11,"""23843""",0,0.0,0.0,"""UNKNOWN""",11,5,11,4,2011,0,0,0,0,0,0.0
2011-11-13,"""23843""",0,0.0,0.0,"""UNKNOWN""",13,7,11,4,2011,1,0,0,0,0,0.0
2011-11-14,"""23843""",0,0.0,0.0,"""UNKNOWN""",14,1,11,4,2011,0,0,0,0,0,0.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2011-11-27,"""23843""",0,0.0,0.0,"""UNKNOWN""",27,7,11,4,2011,1,0,0,0,0,0.0
2011-11-28,"""23843""",0,0.0,0.0,"""UNKNOWN""",28,1,11,4,2011,0,0,0,0,0,0.0
2011-11-29,"""23843""",0,0.0,0.0,"""UNKNOWN""",29,2,11,4,2011,0,0,0,0,0,0.0


In [36]:
display(
    df_gold
    .filter(pl.col("StockCode") == "23843")
    .sort("Week_Start")
    .tail(10)
)

Week_Start,StockCode,Weekly_Demand,Weekly_Revenue,Avg_UnitPrice,Description,Target_Demand_Weekly,Lag_1W,Lag_2W,Lag_4W,Rolling_Mean_4W,Rolling_Std_4W,Month,Quarter,Year,Is_Peak_Season,Demand_Trend_Ratio,Demand_CV,Price_Ratio_Vs_Avg
datetime[μs],str,i64,f64,f64,str,i64,i64,i64,i64,f64,f64,i8,i8,i32,i32,f64,f64,f64
2011-09-26 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,9,3,2011,0,0.0,0.0,0.0
2011-10-03 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,10,4,2011,0,0.0,0.0,0.0
2011-10-10 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,10,4,2011,0,0.0,0.0,0.0
2011-10-17 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,10,4,2011,0,0.0,0.0,0.0
2011-10-24 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,10,4,2011,0,0.0,0.0,0.0
2011-10-31 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,10,4,2011,0,0.0,0.0,0.0
2011-11-07 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,11,4,2011,1,0.0,0.0,0.0
2011-11-14 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,11,4,2011,1,0.0,0.0,0.0
2011-11-21 00:00:00,"""23843""",0,0.0,0.0,"""UNKNOWN""",0,0,0,0,0.0,0.0,11,4,2011,1,0.0,0.0,0.0


In [37]:
silver = pl.read_parquet(
    BASE_DIR / "data" / "2_silver" / "sheet_2010_2011_cleaned.parquet"
)

display(
    silver
    .filter(
        (pl.col("StockCode") == "23843") &
        (pl.col("InvoiceDate").dt.date() == pl.date(2011, 12, 9))
    )
)

Invoice,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
str,str,str,i64,datetime[μs],f64,str,str
"""581483""","""23843""","""PAPER CRAFT , LITTLE BIRDIE""",80995,2011-12-09 09:15:00,2.08,"""16446""","""United Kingdom"""
